In [3]:
# Appendix Table A1: Full Model and Pipeline Comparison

import pandas as pd
from pathlib import Path

# Set this to your actual capstone coding folder
PROJECT_ROOT = Path(r"b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding")

PHASE6_OUTPUT = PROJECT_ROOT / "data" / "processed" / "phase6_models_folds" / "Fold1"
APPENDIX_OUTPUT = PROJECT_ROOT / "data" / "processed" / "appendix_tables"
APPENDIX_OUTPUT.mkdir(parents=True, exist_ok=True)

MODELS = ["pointwise", "pairwise", "lightgbm"]
PIPELINES = ["raw", "global", "per_query"]
DATASETS = ["2007", "2008"]

def find_col(df, possible_names):
    for col in possible_names:
        if col in df.columns:
            return col
    raise ValueError(
        f"None of these columns found: {possible_names}\n"
        f"Available columns: {df.columns.tolist()}"
    )

def make_fail_numeric(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float)
    return series.astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)

print("Looking in:", PHASE6_OUTPUT)

rows = []

for dataset in DATASETS:
    for model in MODELS:
        for pipeline in PIPELINES:
            file_path = PHASE6_OUTPUT / f"{model}_{pipeline}_{dataset}_query_metrics.csv"

            if not file_path.exists():
                print(f"Missing file: {file_path}")
                continue

            df = pd.read_csv(file_path)

            ndcg_col = find_col(df, ["NDCG@5", "ndcg@5", "NDCG@5_primary", "ndcg@5_primary"])
            p5_col = find_col(df, ["P@5_primary", "precision@5", "Precision@5", "P@5"])
            fail_col = find_col(df, ["Failure@5_primary", "failure@5", "Failure@5"])
            rel_col = find_col(df, ["num_relevant_1", "num_relevant"])

            eval_df = df[df[rel_col] > 0].copy()
            fail_numeric = make_fail_numeric(eval_df[fail_col])

            rows.append({
                "Dataset": f"MQ{dataset}",
                "Model": model,
                "Pipeline": pipeline,
                "NDCG@5": eval_df[ndcg_col].mean(),
                "Precision@5": eval_df[p5_col].mean(),
                "Failure@5 (%)": fail_numeric.mean() * 100,
                "Evaluable Queries": len(eval_df)
            })

appendix_table = pd.DataFrame(rows)

appendix_table["NDCG@5"] = appendix_table["NDCG@5"].round(4)
appendix_table["Precision@5"] = appendix_table["Precision@5"].round(4)
appendix_table["Failure@5 (%)"] = appendix_table["Failure@5 (%)"].round(1)

appendix_table = appendix_table.sort_values(
    by=["Dataset", "Model", "Pipeline"]
).reset_index(drop=True)

csv_path = APPENDIX_OUTPUT / "appendix_model_pipeline_comparison.csv"
appendix_table.to_csv(csv_path, index=False)

print("Saved:", csv_path)
display(appendix_table)

Looking in: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds\Fold1
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds\Fold1\pointwise_raw_2008_query_metrics.csv
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds\Fold1\pointwise_global_2008_query_metrics.csv
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds\Fold1\pointwise_per_query_2008_query_metrics.csv
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds\Fold1\pairwise_raw_2008_query_metrics.csv
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds\Fold1\pairwise_global_2008_query_metrics.csv
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds\Fold1\pairwise_per_query_2008_query_

,Dataset,Model,Pipeline,NDCG@5,Precision@5,Failure@5 (%),Evaluable Queries
0,MQ2007,lightgbm,global,0.4984,0.4821,18.3,290
1,MQ2007,lightgbm,per_query,0.5202,0.5103,15.2,290
2,MQ2007,lightgbm,raw,0.4941,0.4917,17.6,290
3,MQ2007,pairwise,global,0.5268,0.5152,17.2,290
4,MQ2007,pairwise,per_query,0.5327,0.5186,17.6,290
5,MQ2007,pairwise,raw,0.5282,0.5145,17.2,290
6,MQ2007,pointwise,global,0.4923,0.4834,15.9,290
7,MQ2007,pointwise,per_query,0.5263,0.5090,19.3,290
8,MQ2007,pointwise,raw,0.4962,0.4890,15.2,290


In [4]:
# Appendix Table B1: Baseline Performance Across K Values

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding")

PHASE6_OUTPUT = PROJECT_ROOT / "data" / "processed" / "phase6_models_folds" / "Fold1"
APPENDIX_OUTPUT = PROJECT_ROOT / "data" / "processed" / "appendix_tables"
APPENDIX_OUTPUT.mkdir(parents=True, exist_ok=True)

file_path = PHASE6_OUTPUT / "pointwise_raw_2007_query_metrics.csv"

df = pd.read_csv(file_path)

# Keep evaluable queries only
df = df[df["num_relevant_1"] > 0].copy()

def fail_to_numeric(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float)
    return series.astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)

rows = []

for k in [1, 3, 5, 10]:
    ndcg_col_options = [f"NDCG@{k}", f"ndcg@{k}", f"NDCG@{k}_primary", f"ndcg@{k}_primary"]
    p_col_options = [f"P@{k}_primary", f"Precision@{k}", f"precision@{k}", f"P@{k}"]
    fail_col_options = [f"Failure@{k}_primary", f"Failure@{k}", f"failure@{k}"]

    ndcg_col = next((c for c in ndcg_col_options if c in df.columns), None)
    p_col = next((c for c in p_col_options if c in df.columns), None)
    fail_col = next((c for c in fail_col_options if c in df.columns), None)

    if ndcg_col is None or p_col is None or fail_col is None:
        print(f"Missing one or more columns for K={k}")
        print("Available columns:", df.columns.tolist())
        continue

    rows.append({
        "K": k,
        "NDCG@K": round(df[ndcg_col].mean(), 4),
        "Precision@K": round(df[p_col].mean(), 4),
        "Failure@K (%)": round(fail_to_numeric(df[fail_col]).mean() * 100, 1),
        "Evaluable Queries": len(df)
    })

k_table = pd.DataFrame(rows)

csv_path = APPENDIX_OUTPUT / "appendix_baseline_performance_across_k.csv"
k_table.to_csv(csv_path, index=False)

print("Saved:", csv_path)
display(k_table)

Saved: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\appendix_tables\appendix_baseline_performance_across_k.csv


,K,NDCG@K,Precision@K,Failure@K (%),Evaluable Queries
0,1,0.4747,0.5345,46.6,290
1,3,0.4816,0.5034,25.2,290
2,5,0.4962,0.4890,15.2,290
3,10,0.5284,0.4531,8.3,290


In [5]:
# Appendix Table C1: Statistical Test Outputs

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding")

PHASE7_OUTPUT = PROJECT_ROOT / "data" / "processed" / "phase7_statistical_tests"
APPENDIX_OUTPUT = PROJECT_ROOT / "data" / "processed" / "appendix_tables"
APPENDIX_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Looking in:", PHASE7_OUTPUT)

# Try common Phase 7 file names
files = {
    "NDCG@5": PHASE7_OUTPUT / "phase7_stats_ndcg_vs_baseline.csv",
    "Precision@5": PHASE7_OUTPUT / "phase7_stats_p5_vs_baseline.csv",
    "Failure@5": PHASE7_OUTPUT / "phase7_stats_failure_vs_baseline.csv",
}

def find_col(df, options):
    for col in options:
        if col in df.columns:
            return col
    return None

all_rows = []

for metric_name, file_path in files.items():
    if not file_path.exists():
        print(f"Missing file: {file_path}")
        continue

    df = pd.read_csv(file_path)
    print(f"\nLoaded {metric_name}:")
    print(df.columns.tolist())

    dataset_col = find_col(df, ["dataset", "Dataset"])
    config_col = find_col(df, ["config", "comparison", "model_config", "candidate_config", "model_pipeline"])
    p_col = find_col(df, ["pval_raw", "p_value", "pvalue", "pval"])
    q_col = find_col(df, ["qval_fdr", "q_value", "qvalue", "qval"])
    supported_col = find_col(df, ["supported", "is_supported", "significant", "fdr_significant"])

    # Effect size columns may differ by metric
    effect_col = find_col(df, [
        "cliffs_delta",
        "cliff_delta",
        "risk_diff",
        "risk_difference",
        "effect_size"
    ])

    # Test name based on metric
    if metric_name in ["NDCG@5", "Precision@5"]:
        test_name = "Wilcoxon signed-rank"
    else:
        test_name = "McNemar"

    for _, row in df.iterrows():
        all_rows.append({
            "Dataset": row[dataset_col] if dataset_col else "Unknown",
            "Metric": metric_name,
            "Comparison": row[config_col] if config_col else "Unknown",
            "Test": test_name,
            "Raw p-value": row[p_col] if p_col else None,
            "FDR q-value": row[q_col] if q_col else None,
            "Effect Size": row[effect_col] if effect_col else None,
            "Supported After FDR": row[supported_col] if supported_col else None
        })

stats_table = pd.DataFrame(all_rows)

# Clean formatting
for col in ["Raw p-value", "FDR q-value", "Effect Size"]:
    if col in stats_table.columns:
        stats_table[col] = pd.to_numeric(stats_table[col], errors="coerce").round(4)

# Keep MQ2007 only for appendix clarity if available
if "Dataset" in stats_table.columns:
    stats_table["Dataset"] = stats_table["Dataset"].astype(str)
    mq2007_mask = stats_table["Dataset"].str.contains("2007", case=False, na=False)
    if mq2007_mask.any():
        stats_table = stats_table[mq2007_mask].copy()

stats_table = stats_table.reset_index(drop=True)

csv_path = APPENDIX_OUTPUT / "appendix_statistical_test_summary.csv"
stats_table.to_csv(csv_path, index=False)

print("\nSaved:", csv_path)
display(stats_table)

Looking in: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase7_statistical_tests
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase7_statistical_tests\phase7_stats_ndcg_vs_baseline.csv
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase7_statistical_tests\phase7_stats_p5_vs_baseline.csv
Missing file: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase7_statistical_tests\phase7_stats_failure_vs_baseline.csv

Saved: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\appendix_tables\appendix_statistical_test_summary.csv


""


In [7]:
# Appendix Table C1: Statistical Test Summary vs Baseline

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding")
APPENDIX_OUTPUT = PROJECT_ROOT / "data" / "processed" / "appendix_tables"
APPENDIX_OUTPUT.mkdir(parents=True, exist_ok=True)

data = [
    # Failure@5 vs baseline, MQ2007
    ["Failure@5", "lightgbm", "global", "McNemar", 0.031, 0.175, 0.436, "No"],
    ["Failure@5", "lightgbm", "per_query", "McNemar", 0.000, 1.000, 1.000, "No"],
    ["Failure@5", "lightgbm", "raw", "McNemar", 0.024, 0.296, 0.436, "No"],
    ["Failure@5", "pairwise", "global", "McNemar", 0.021, 0.327, 0.436, "No"],
    ["Failure@5", "pairwise", "per_query", "McNemar", 0.024, 0.281, 0.436, "No"],
    ["Failure@5", "pairwise", "raw", "McNemar", 0.021, 0.327, 0.436, "No"],
    ["Failure@5", "pointwise", "global", "McNemar", 0.007, 0.500, 0.571, "No"],
    ["Failure@5", "pointwise", "per_query", "McNemar", 0.041, 0.036, 0.286, "No"],

    # NDCG@5 vs baseline, MQ2007
    ["NDCG@5", "lightgbm", "global", "Wilcoxon", -0.007, 0.814, 0.866, "No"],
    ["NDCG@5", "lightgbm", "per_query", "Wilcoxon", 0.079, 0.104, 0.160, "No"],
    ["NDCG@5", "lightgbm", "raw", "Wilcoxon", -0.003, 0.866, 0.866, "No"],
    ["NDCG@5", "pairwise", "global", "Wilcoxon", 0.093, 0.008, 0.016, "Yes"],
    ["NDCG@5", "pairwise", "per_query", "Wilcoxon", 0.128, 0.005, 0.015, "Yes"],
    ["NDCG@5", "pairwise", "raw", "Wilcoxon", 0.103, 0.005, 0.015, "Yes"],
    ["NDCG@5", "pointwise", "global", "Wilcoxon", -0.024, 0.120, 0.160, "No"],
    ["NDCG@5", "pointwise", "per_query", "Wilcoxon", 0.114, 0.004, 0.015, "Yes"],

    # Precision@5 vs baseline, MQ2007
    ["Precision@5", "lightgbm", "global", "Wilcoxon", -0.028, 0.824, 0.834, "No"],
    ["Precision@5", "lightgbm", "per_query", "Wilcoxon", 0.055, 0.192, 0.281, "No"],
    ["Precision@5", "lightgbm", "raw", "Wilcoxon", 0.021, 0.834, 0.834, "No"],
    ["Precision@5", "pairwise", "global", "Wilcoxon", 0.093, 0.014, 0.056, "No"],
    ["Precision@5", "pairwise", "per_query", "Wilcoxon", 0.083, 0.008, 0.056, "No"],
    ["Precision@5", "pairwise", "raw", "Wilcoxon", 0.093, 0.022, 0.059, "No"],
    ["Precision@5", "pointwise", "global", "Wilcoxon", -0.031, 0.211, 0.281, "No"],
    ["Precision@5", "pointwise", "per_query", "Wilcoxon", 0.059, 0.045, 0.090, "No"],
]

stats_table = pd.DataFrame(
    data,
    columns=[
        "Metric",
        "Model",
        "Pipeline",
        "Test",
        "Effect Size",
        "Raw p-value",
        "FDR q-value",
        "Significant After FDR"
    ]
)

csv_path = APPENDIX_OUTPUT / "appendix_statistical_test_summary.csv"
stats_table.to_csv(csv_path, index=False)

print("Saved:", csv_path)

# Copies clean table to clipboard for Word
stats_table.to_clipboard(index=False)

display(stats_table)

Saved: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\appendix_tables\appendix_statistical_test_summary.csv


,Metric,Model,Pipeline,Test,Effect Size,Raw p-value,FDR q-value,Significant After FDR
0,Failure@5,lightgbm,global,McNemar,0.031,0.175,0.436,No
1,Failure@5,lightgbm,per_query,McNemar,0.000,1.000,1.000,No
2,Failure@5,lightgbm,raw,McNemar,0.024,0.296,0.436,No
3,Failure@5,pairwise,global,McNemar,0.021,0.327,0.436,No
4,Failure@5,pairwise,per_query,McNemar,0.024,0.281,0.436,No
5,Failure@5,pairwise,raw,McNemar,0.021,0.327,0.436,No
6,Failure@5,pointwise,global,McNemar,0.007,0.500,0.571,No
7,Failure@5,pointwise,per_query,McNemar,0.041,0.036,0.286,No
8,NDCG@5,lightgbm,global,Wilcoxon,-0.007,0.814,0.866,No
9,NDCG@5,lightgbm,per_query,Wilcoxon,0.079,0.104,0.160,No


In [8]:
# Appendix Table D1: Persistent vs Successful Query Comparison

import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product

PROJECT_ROOT = Path(r"b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding")

PHASE6_OUTPUT = PROJECT_ROOT / "data" / "processed" / "phase6_models_folds" / "Fold1"
APPENDIX_OUTPUT = PROJECT_ROOT / "data" / "processed" / "appendix_tables"
APPENDIX_OUTPUT.mkdir(parents=True, exist_ok=True)

MODELS = ["pointwise", "pairwise", "lightgbm"]
PIPELINES = ["raw", "global", "per_query"]
DATASET = "2007"
BASELINE_KEY = "pointwise_raw_2007"

def make_fail_flag(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).round().astype(int).clip(0, 1)
    return series.astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)

# --------------------------------------------------
# Load all query metric files and reconstruct groups
# --------------------------------------------------

evaluable_sets = {}
failure_sets = {}

for model, pipeline in product(MODELS, PIPELINES):
    key = f"{model}_{pipeline}_{DATASET}"
    file_path = PHASE6_OUTPUT / f"{key}_query_metrics.csv"
    
    qm = pd.read_csv(file_path)
    
    evaluable_sets[key] = set(qm.loc[qm["num_relevant_1"] > 0, "qid"])
    
    fail_flag = make_fail_flag(qm["Failure@5_primary"])
    fail_mask = (qm["num_relevant_1"] > 0) & (fail_flag == 1)
    failure_sets[key] = set(qm.loc[fail_mask, "qid"])

all_evaluable = set.intersection(*evaluable_sets.values())
all_failing = set.union(*failure_sets.values())
persistent_qids = set.intersection(*failure_sets.values())
successful_qids = all_evaluable - all_failing
non_persistent_qids = all_failing - persistent_qids

# --------------------------------------------------
# Load baseline query metrics and predictions
# --------------------------------------------------

baseline_qm = pd.read_csv(PHASE6_OUTPUT / f"{BASELINE_KEY}_query_metrics.csv")
pred = pd.read_csv(PHASE6_OUTPUT / f"{BASELINE_KEY}_predictions.csv")

pred["score"] = pd.to_numeric(pred["score"])
pred["label"] = pd.to_numeric(pred["label"])

def compute_score_gap(qid):
    q_docs = pred[pred["qid"] == qid].copy()
    
    if len(q_docs) == 0:
        return np.nan
    
    q_docs = q_docs.sort_values("score", ascending=False).reset_index(drop=True)
    relevant_docs = q_docs[q_docs["label"] >= 1]
    
    if len(relevant_docs) == 0:
        return np.nan
    
    best_relevant_score = relevant_docs["score"].max()
    k_actual = min(5, len(q_docs))
    score_at_rank5 = q_docs.loc[k_actual - 1, "score"]
    
    return best_relevant_score - score_at_rank5

def summarize_group(group_name, qids):
    sub = baseline_qm[baseline_qm["qid"].isin(qids)].copy()
    
    score_gaps = [compute_score_gap(qid) for qid in sub["qid"]]
    
    return {
        "Group": group_name,
        "Queries": len(sub),
        "Percent of Evaluable (%)": round(len(sub) / len(all_evaluable) * 100, 1),
        "Mean Relevant Docs": round(sub["num_relevant_1"].mean(), 2),
        "% With Exactly 1 Relevant": round((sub["num_relevant_1"] == 1).mean() * 100, 1),
        "Mean Score Gap": round(np.nanmean(score_gaps), 4)
    }

summary_rows = [
    summarize_group("Persistent Failures", persistent_qids),
    summarize_group("Non-Persistent Failures", non_persistent_qids),
    summarize_group("Successful Queries", successful_qids),
]

persistent_table = pd.DataFrame(summary_rows)

csv_path = APPENDIX_OUTPUT / "appendix_persistent_failure_analysis.csv"
persistent_table.to_csv(csv_path, index=False)

print("Saved:", csv_path)

# Copy clean table to clipboard for Word
persistent_table.to_clipboard(index=False)

display(persistent_table)

Saved: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\appendix_tables\appendix_persistent_failure_analysis.csv


,Group,Queries,Percent of Evaluable (%),Mean Relevant Docs,% With Exactly 1 Relevant,Mean Score Gap
0,Persistent Failures,22,7.6,2.36,50.0,-0.1722
1,Non-Persistent Failures,63,21.7,5.92,15.9,0.0104
2,Successful Queries,205,70.7,16.32,4.9,0.1428


In [9]:
# Appendix Table E1: Synthetic Stress Test Results

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding")

PHASE12_OUTPUT = PROJECT_ROOT / "data" / "processed" / "phase12_synthetic_stress"
APPENDIX_OUTPUT = PROJECT_ROOT / "data" / "processed" / "appendix_tables"
APPENDIX_OUTPUT.mkdir(parents=True, exist_ok=True)

summary_file = PHASE12_OUTPUT / "phase12_sparsity_stress_summary.csv"

stress_df = pd.read_csv(summary_file)

# Keep expected order
level_order = ["original", "cap_rel_3", "cap_rel_2", "cap_rel_1"]
label_map = {
    "original": "Original",
    "cap_rel_3": "Cap Relevant Docs @ 3",
    "cap_rel_2": "Cap Relevant Docs @ 2",
    "cap_rel_1": "Cap Relevant Docs @ 1",
}

stress_df["sparsity_level"] = pd.Categorical(
    stress_df["sparsity_level"],
    categories=level_order,
    ordered=True
)

stress_df = stress_df.sort_values("sparsity_level").reset_index(drop=True)

# Build compact appendix table
table_e = pd.DataFrame({
    "Sparsity Condition": stress_df["sparsity_level"].map(label_map),
    "Mean Failure@5 (%)": stress_df["mean_failure_pct"].round(1),
    "Std. Dev.": stress_df["std_failure_rate"].mul(100).round(2),
})

# Add delta vs original
original_failure = table_e.loc[0, "Mean Failure@5 (%)"]
table_e["Change vs Original (pp)"] = (
    table_e["Mean Failure@5 (%)"] - original_failure
).round(1)

csv_path = APPENDIX_OUTPUT / "appendix_synthetic_stress_test.csv"
table_e.to_csv(csv_path, index=False)

print("Saved:", csv_path)

# Copy clean table to clipboard for Word
table_e.to_clipboard(index=False)

display(table_e)

Saved: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\appendix_tables\appendix_synthetic_stress_test.csv


,Sparsity Condition,Mean Failure@5 (%),Std. Dev.,Change vs Original (pp)
0,Original,15.2,0.00,0.0
1,Cap Relevant Docs @ 3,48.7,2.18,33.5
2,Cap Relevant Docs @ 2,59.2,2.14,44.0
3,Cap Relevant Docs @ 1,73.9,1.94,58.7
